In [1]:
from pynq.overlays.base import BaseOverlay
import threading
import time
from pynq.lib.arduino import Arduino_Analog
import socket
import os
import random
import json
import numpy
import cairo
import math

base = BaseOverlay("base.bit")
base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds

In [2]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn off all GPIO pins on a PMOD
void reset_gpio(){
    for (unsigned int i = 0; i < 8; i++)
    {
        gpio pin_out = gpio_open(i);
        gpio_set_direction(pin_out, GPIO_OUT);
        gpio_write(pin_out, 0);
    }
}

//Function to turn on/off a selected pin of PMODB
unsigned int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

In [3]:
reset_gpio()
read_gpio(1)

0

In [4]:
def map_val(value, fromLow, fromHigh, toLow, toHigh):
    return (value - fromLow) * (toHigh - toLow) / (fromHigh - fromLow) + toLow

def direction(x, y):
        flag = 0
        if x > 0.55:
            flag = 1
        if x < -.55:
            flag = 2
        if y > .55:
            flag = 3
        if y < -.55:
            flag = 4
        return flag

In [5]:
def mapGenerate(coord):
    
    width = 128 # map will always be a square, so only one dimension is needed

    data = numpy.zeros((width, width, 4), dtype=numpy.uint8)
    surface = cairo.ImageSurface.create_for_data(
        data, cairo.FORMAT_ARGB32, width, width)
    cr = cairo.Context(surface)

    # fill with solid white

    cr.set_source_rgb(1.0, 1.0, 1.0)
    cr.paint()

    # draw green radius (outer zone)
    print("Generating green zone...")
    cr.arc(coord[0], coord[1], width/4, 0, 2*math.pi) #arc(xcoord, ycoord, radius, ??, circle)
    cr.set_source_rgb(0.0, 1.0, 0.0) #green
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 128:
                data[i,j,0] = 128

    # draw yellow radius (middle zone)
    print("Generating yellow zone...")
    cr.arc(coord[0], coord[1], width/7, 0, 2*math.pi) 
    cr.set_source_rgb(1.0, 1.0, 0.0) #yellow
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 64:
                data[i,j,0] = 64

    # draw red radius (inner zone)
    print("Generating red zone...")
    cr.arc(coord[0], coord[1], width/25, 0, 2*math.pi) 
    cr.set_source_rgb(1.0, 0.0, 0.0) #red
    cr.fill()

    for i in range(width):
        for j in range(width):
            if data[i,j,0] < 32:
                data[i,j,0] = 32

    # draw dig radius
    print("Generating dig zone...")
    cr.arc(coord[0], coord[1], 2, 0, 2*math.pi) 
    cr.set_source_rgb(0.0, 0.0, 0.0) # black
    cr.fill()

    # write output
    print(coord)
    print(data[(coord[1]-8):(coord[1]+8), (coord[0]-7):(coord[0]+7), 0])
    surface.write_to_png("circle_complete_2.png")
    return data

In [10]:
def gameLoop(_treasureCoord, _gameStart, _gameCon, _gameWin):
    
    print("Waiting on game to start...")
    
    analog_stick = Arduino_Analog(base.ARDUINO, [0, 1])
    
    data = mapGenerate(_treasureCoord)
    
    y = 10
    x = 10
    prox = "null"
    win = False
    lose = False
    blink = False

    max_x = 127
    max_y = 127
    
    while win == False and lose == False:  
        if (_gameCon.is_set()):
            lose = True

        else:
            vals = analog_stick.read() 

            x_val = vals[0]
            y_val = vals[1]

            y_val_map = map_val(y_val, 0, 3.3, 1, -1)
            x_val_map = map_val(x_val, 0, 3.3, -1, 1)

            y_val_map = round(y_val_map, 2)
            x_val_map = round(x_val_map, 2)

            direc = direction(x_val_map, y_val_map)

            coordinatex = 100
            coordinatey = 100

            if direc == 1:
                x = x + 1
                time.sleep(.1)
            if direc == 2:
                x = x - 1
                time.sleep(.1)
            if direc == 3:
                y = y - 1
                time.sleep(.1)
            if direc == 4:
                y = y + 1
                time.sleep(.1)

            x = max(0,min(x,max_x))
            y = max(0,min(y,max_y))

            if data[y,x,0] == 255:
                prox = "BLANK "
                write_gpio(3,0)
                write_gpio(2,0)

            if data[y,x,0] < 255 and data[y,x,0] > 127:
                prox = "GREEN "
                write_gpio(3,1)
                write_gpio(2,0)

            if data[y,x,0] < 128 and data[y,x,0] > 63:
                prox = "YELLOW"
                write_gpio(3,1)
                write_gpio(2,1)

            if data[y,x,0] < 64 and data[y,x,0] > 31:
                prox = "RED   "
                write_gpio(3,0)
                write_gpio(2,1)

            if data[y,x,0] < 32:
                prox = "DIG   "
                blink = not blink
                if blink:
                    write_gpio(2,1)
                if not blink:
                    write_gpio(2,0)
                if (btns[0].read() == 1):
                    win = True
                    _gameWin.set()

            print(f"X: {x:2d} Y: {y:2d} Proximity: {prox}", end="\r")
            time.sleep(.1)
            ##except: # receive winning msg from other player
            ##   print("You lost! Game over...")
            
    if win == True:
        print("You found the treasure!!")
        write_gpio(2,0)
        for i in range(5):
            write_gpio(1,1)
            time.sleep(0.75)
            write_gpio(1,0)
            time.sleep(0.75)
    else:
        print("You lost! Game over...")
        write_gpio(2,0)
        for i in range(5):
            write_gpio(2,1)
            time.sleep(1)
            write_gpio(2,0)
            time.sleep(1)

In [9]:
def networkProc(_teasureCoord, _gameStart, _gameCon, _gameWin):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind(('192.168.8.174',12345))
    sock.listen()
    print("Waiting for connection...")
    (clientSock,addr) = sock.accept()
    print("Connection received!")
    rec = clientSock.recv(1024)
    if rec:
        print(type(rec))
        print(f'data received: {rec}')
        clientSock.sendall(b'ACK: message sent successfully')
    print(json.loads(rec))
    _treasureCoord = json.loads(rec)
    _gameStart.set()
    while (not(_gameCon.is_set())):
        if (_gameWin.is_set()):
            msg = '1'
            b_en = msg.encode(encoding= "utf-8")
            clientSock.send(b_en)
            _gameCon.set()
        condi = clientSock.recv(1024)
        dec = condi.decode()
        if dec == '1':
            _gameCon.set()
            
    ##gameMap = mapGenerate(json.loads(rec))
    ##gameLoop(gameMap, clientSock)
    print("Success!")
    sock.close()
    print("Server closed. See ya!")

def clientProc():
    print("Press 1 to Connect to Faisal's Server!")
    while (btns[0].read() == 0):
        time.sleep(0.01)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    print("Attempting to Connect to Server...")
    sock.connect(('192.168.8.222',12345))
    msg = 'j'
    b_en = json.dumps(msg).encode("utf-8")
    sock.send(b_en)
    print("Signal sent! Did they get it?")
    time.sleep(1)
    sock.close()
    print("Client closed!")

In [8]:
threads = [] # a future list of all our processes
gameStart = threading.Event()
gameCon = threading.Event()
gameWin = threading.Event()
treasureCoord = [0,0]

# Launch process1
t1 = threading.Thread(target=networkProc, args=(teasureCoord, gameStart, gameCon, gameWin))
threads.append(t1)
t1.start() # start the thread
print("Networking started!")


# Launch process2
t2 = threading.Thread(target=gameLoop, args=(teasureCoord, gameStart, gameCon, gameWin))
t2.start() # start the process
procs.append(t2)

Server started!
Waiting for connection...
Connection received!
<class 'bytes'>
data received: b'[67, 102]'
Generating green zone...
Generating yellow zone...
Generating red zone...
Generating dig zone...
[67, 102]
[[64 64 64 64 64 64 64 64 64 64 64 64 64 64]
 [64 64 64 64 64 64 64 64 64 64 64 64 64 64]
 [64 64 64 64 64 64 58 58 64 64 64 64 64 64]
 [64 64 64 62 32 32 32 32 32 32 62 64 64 64]
 [64 64 61 32 32 32 32 32 32 32 32 61 64 64]
 [64 64 32 32 32 32 32 32 32 32 32 32 64 64]
 [64 64 32 32 32 22  3  3 22 32 32 32 64 64]
 [64 59 32 32 32  3  0  0  3 32 32 32 59 64]
 [64 59 32 32 32  3  0  0  3 32 32 32 60 64]
 [64 64 32 32 32 22  3  3 22 32 32 32 64 64]
 [64 64 32 32 32 32 32 32 32 32 32 32 64 64]
 [64 64 61 32 32 32 32 32 32 32 32 61 64 64]
 [64 64 64 62 32 32 32 32 32 32 62 64 64 64]
 [64 64 64 64 64 64 59 59 64 64 64 64 64 64]
 [64 64 64 64 64 64 64 64 64 64 64 64 64 64]
 [64 64 64 64 64 64 64 64 64 64 64 64 64 64]]


Process Process-1:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_27688/3699240864.py", line 15, in serverProc
    gameLoop(gameMap, clientSock)
  File "/tmp/ipykernel_27688/1165188433.py", line 21, in gameLoop
    rec = sock.recv(1024)
KeyboardInterrupt
